In [ ]:
import numpy as np, pandas as pd, os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Experiment 3b — Accuracy-Weighted RSA

Tests the report's proposed composite metric that weights RSA by task accuracy, to check whether the non-monotonic opacity hypothesis holds.

In [ ]:
!pip install torch matplotlib scipy -q

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = ''
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from scipy.stats import spearmanr
from scipy.spatial.distance import pdist
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
torch.manual_seed(0); np.random.seed(0)

N_FEATURES, N_CANDIDATES = 16, 10
N_TRAIN, N_VAL = 8000, 2000
VOCAB_SIZE, MSG_LEN, HIDDEN = 32, 2, 128
N_EPOCHS, BATCH_SIZE, LR = 80, 128, 1e-3

def make_dataset(n, seed):
    rng = np.random.default_rng(seed)
    objs = rng.integers(0, 2, (n, N_CANDIDATES, N_FEATURES)).astype(np.float32)
    return TensorDataset(torch.tensor(objs[:,0,:]), torch.tensor(objs), torch.zeros(n,dtype=torch.long))

train_ds = make_dataset(N_TRAIN, 1)
val_ds   = make_dataset(N_VAL, 2)

class Sender(nn.Module):
    def __init__(self):
        super().__init__()
        self.trunk = nn.Sequential(nn.Linear(N_FEATURES,HIDDEN), nn.ReLU(), nn.Linear(HIDDEN,HIDDEN), nn.ReLU())
        self.heads = nn.ModuleList([nn.Linear(HIDDEN, VOCAB_SIZE) for _ in range(MSG_LEN)])
    def forward(self, x):
        h = self.trunk(x)
        return [head(h) for head in self.heads]

class Receiver(nn.Module):
    def __init__(self):
        super().__init__()
        self.msg_proj = nn.Sequential(nn.Linear(VOCAB_SIZE*MSG_LEN, HIDDEN), nn.ReLU())
        self.obj_proj = nn.Sequential(nn.Linear(N_FEATURES, HIDDEN), nn.ReLU(), nn.Linear(HIDDEN, HIDDEN))
    def forward(self, msgs, cands):
        h = self.msg_proj(torch.cat(msgs, dim=-1))
        o = self.obj_proj(cands)
        return torch.bmm(o, h.unsqueeze(-1)).squeeze(-1)

def train_opacity(tau, continuous=False):
    sender = Sender(); receiver = Receiver()
    opt = torch.optim.Adam(list(sender.parameters())+list(receiver.parameters()), lr=LR)
    for epoch in range(N_EPOCHS):
        sender.train(); receiver.train()
        for si, cands, labels in DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True):
            logits_list = sender(si)
            if continuous:
                msgs = [torch.softmax(lg/tau, dim=-1) for lg in logits_list]
            else:
                msgs = [F.gumbel_softmax(lg, tau=tau, hard=True) for lg in logits_list]
            loss = F.cross_entropy(receiver(msgs, cands), labels)
            loss.backward(); opt.step(); opt.zero_grad()
    return sender, receiver

def eval_and_messages(sender, receiver, continuous=False):
    sender.eval(); receiver.eval()
    accs, all_msgs, all_objs = [], [], []
    with torch.no_grad():
        for si, cands, labels in DataLoader(val_ds, batch_size=256):
            logits_list = sender(si)
            if continuous:
                msgs = [torch.softmax(lg, dim=-1) for lg in logits_list]
                msg_repr = torch.cat(msgs, dim=-1)
            else:
                toks = [lg.argmax(-1) for lg in logits_list]
                msgs = [F.one_hot(t, VOCAB_SIZE).float() for t in toks]
                msg_repr = torch.stack(toks, dim=-1).float()
            scores = receiver(msgs, cands)
            accs.append((scores.argmax(-1)==labels).float().mean().item())
            all_msgs.append(msg_repr.numpy()); all_objs.append(si.numpy())
    return float(np.mean(accs)), np.vstack(all_msgs), np.vstack(all_objs)

def compute_rsa(msgs, objs, n=400):
    idx = np.random.default_rng(0).choice(len(msgs), min(n,len(msgs)), replace=False)
    md = pdist(msgs[idx], 'hamming' if msgs.shape[1]<=MSG_LEN else 'cosine')
    od = pdist(objs[idx], 'hamming')
    rho, _ = spearmanr(md, od)
    return float(rho) if not np.isnan(rho) else 0.0

def compute_topsim(msgs, objs, n=400):
    idx = np.random.default_rng(1).choice(len(msgs), min(n,len(msgs)), replace=False)
    rho, _ = spearmanr(pdist(msgs[idx],'hamming'), pdist(objs[idx],'cityblock'))
    return float(rho) if not np.isnan(rho) else 0.0

def symbol_entropy(msgs):
    vals, counts = np.unique(msgs[:,0].astype(int), return_counts=True)
    p = counts/counts.sum()
    return float(-(p*np.log(p+1e-9)).sum())

def is_non_monotonic(values):
    peak = int(np.argmax(values))
    return 0 < peak < len(values)-1

print('Setup complete')

In [ ]:
CONDITIONS = [('continuous', 99.0, True)] +     [(f'gs_tau={t}', t, False) for t in [10, 5, 2, 1, 0.5, 0.3, 0.1]]

results = []
for name, tau, continuous in CONDITIONS:
    print(f'Training {name}...')
    s, r = train_opacity(tau, continuous)
    acc, msgs, objs = eval_and_messages(s, r, continuous)
    rsa    = compute_rsa(msgs, objs)
    topsim = compute_topsim(msgs, objs)
    ent    = symbol_entropy(msgs)
    aw_rsa    = rsa * acc
    aw_topsim = topsim * acc
    results.append(dict(name=name, tau=tau, continuous=continuous,
                        accuracy=acc, rsa=rsa, topsim=topsim,
                        aw_rsa=aw_rsa, aw_topsim=aw_topsim, entropy=ent))
    print(f'  acc={acc:.3f}  RSA={rsa:.3f}  topsim={topsim:.3f}  AW-RSA={aw_rsa:.3f}')

gs = sorted([r for r in results if not r['continuous']], key=lambda r: -r['tau'])
print('\nNON-MONOTONICITY TEST:')
for metric in ['rsa','topsim','aw_rsa','aw_topsim','accuracy']:
    vals = [r[metric] for r in gs]
    nm = is_non_monotonic(vals)
    peak = gs[int(np.argmax(vals))]['name']
    print(f'  {metric:<12}: peak={peak:<12} {"NON-MONOTONIC" if nm else "monotonic"}')

In [ ]:
gs = sorted([r for r in results if not r['continuous']], key=lambda r: -r['tau'])
x = range(len(gs))
labels = [r['name'].replace('gs_tau=','τ=') for r in gs]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
ax.plot(x, [r['rsa'] for r in gs], 'o-', color='#378ADD', label='plain RSA')
ax.plot(x, [r['topsim'] for r in gs], 's-', color='#7F77DD', label='topsim')
ax.plot(x, [r['accuracy'] for r in gs], '^-', color='#1D9E75', label='accuracy')
ax.set_xticks(list(x)); ax.set_xticklabels(labels, rotation=45)
ax.set_title('Original metrics (RSA monotonic,\naccuracy peaks in the middle)')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(x, [r['aw_rsa'] for r in gs], 'o-', color='#E24B4A', label='AW-RSA', linewidth=2.5)
ax.plot(x, [r['aw_topsim'] for r in gs], 's-', color='#EF9F27', label='AW-topsim', linewidth=2.5)
peak_i = int(np.argmax([r['aw_rsa'] for r in gs]))
ax.axvline(peak_i, color='gray', linestyle='--', alpha=0.6, label=f'AW-RSA peak: {labels[peak_i]}')
ax.set_xticks(list(x)); ax.set_xticklabels(labels, rotation=45)
ax.set_title("Accuracy-weighted symbol quality\n(report's proposed metric)")
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[2]
ax.plot(x, [r['entropy'] for r in gs], 'o-', color='#D4537E')
ax.set_xticks(list(x)); ax.set_xticklabels(labels, rotation=45)
ax.set_ylabel('Symbol entropy')
ax.set_title('Symbol entropy collapse at low τ\n(explains accuracy drop at extremes)')
ax.grid(True, alpha=0.3)

plt.suptitle('Exp 3b — Accuracy-Weighted RSA Opacity Sweep\n'
             'Does symbol quality peak at INTERMEDIATE opacity when weighted by accuracy?',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('exp3b_accuracy_weighted_rsa.png', dpi=150, bbox_inches='tight')
plt.show()

aw_nm = is_non_monotonic([r['aw_rsa'] for r in gs])
print('\nReport prediction: AW-RSA should be NON-MONOTONIC (interior peak)')
if aw_nm:
    print('VERDICT: CONFIRMED — accuracy-weighted RSA has an interior peak.')
    print('The non-monotonic opacity hypothesis holds for the useful symbol-quality metric.')
else:
    print('VERDICT: NOT CONFIRMED — AW-RSA peak is at an edge. Try more epochs or MSG_LEN=1.')